In [115]:
import pandas as pd
from pathlib import Path
import sys
RAIZ = Path.cwd().resolve()  # rode o notebook a partir da raiz do repo
sys.path.insert(0, str(RAIZ / "src"))
sys.path.insert(0, str(RAIZ))
from ts_utils import BASELINES, M, SERIES, calcular_baselines, calcular_metricas, carregar_dados

from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error


import joblib

In [102]:
df_treino, df_val = carregar_dados(RAIZ)

In [103]:
df_treino

,date,store_total,FOODS,HOBBIES
0,2011-01-29,4337,3239,556
1,2011-01-30,4155,3137,498
2,2011-01-31,2816,2008,415
3,2011-02-01,3051,2258,392
4,2011-02-02,2630,2032,268
...,...,...,...,...
1880,2016-03-23,3770,2554,454
1881,2016-03-24,3970,2681,474
1882,2016-03-25,4904,3421,446
1883,2016-03-26,6139,4251,610


In [104]:
df_treino.dtypes

date           datetime64[us]
store_total             int64
FOODS                   int64
HOBBIES                 int64
dtype: object

In [105]:
df_treino_interno = df_treino[df_treino['date'] <='2016-02-28' ]
df_validacao_interno = df_treino[df_treino['date'] > '2016-02-28' ]

In [106]:
df_covariaveis = pd.read_csv('dados/covariaveis.csv')
df_covariaveis['date'] = pd.to_datetime(df_covariaveis['date'])


# Colocando os dummies
df_covariaveis = pd.get_dummies(df_covariaveis, columns= ['event_name_1','event_type_1','event_name_2','event_type_2'])

mask_treino = df_covariaveis['date'] <= '2016-02-28'

# Calcula a média e desvio APENAS no treino
media_foods = df_covariaveis.loc[mask_treino, 'preco_FOODS'].mean()
desvio_foods = df_covariaveis.loc[mask_treino, 'preco_FOODS'].std()

# Aplica a padronização na coluna inteira (treino e validação)
df_covariaveis['preco_FOODS_padronizada'] = (df_covariaveis['preco_FOODS'] - media_foods) / desvio_foods

In [107]:
df_resultado_treino = pd.merge(df_treino_interno,df_covariaveis, on= 'date', how = 'inner' )

In [108]:
df_resultado_treino


,date,store_total,FOODS,HOBBIES,snap_CA,preco_store_total,preco_FOODS,preco_HOBBIES,event_name_1_Chanukah End,event_name_1_Christmas,...,event_type_1_National,event_type_1_Religious,event_type_1_Sporting,event_name_2_Cinco De Mayo,event_name_2_Easter,event_name_2_Father's day,event_name_2_OrthodoxEaster,event_type_2_Cultural,event_type_2_Religious,preco_FOODS_padronizada
0,2011-01-29,4337,3239,556,0,4.0811,3.1492,4.4688,False,False,...,False,False,False,False,False,False,False,False,False,-2.165717
1,2011-01-30,4155,3137,498,0,4.0811,3.1492,4.4688,False,False,...,False,False,False,False,False,False,False,False,False,-2.165717
2,2011-01-31,2816,2008,415,0,4.0811,3.1492,4.4688,False,False,...,False,False,False,False,False,False,False,False,False,-2.165717
3,2011-02-01,3051,2258,392,1,4.0811,3.1492,4.4688,False,False,...,False,False,False,False,False,False,False,False,False,-2.165717
4,2011-02-02,2630,2032,268,1,4.0811,3.1492,4.4688,False,False,...,False,False,False,False,False,False,False,False,False,-2.165717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1852,2016-02-24,3369,2076,473,0,4.4950,3.3470,5.5897,False,False,...,False,False,False,False,False,False,False,False,False,1.272119
1853,2016-02-25,3396,2176,451,0,4.4950,3.3470,5.5897,False,False,...,False,False,False,False,False,False,False,False,False,1.272119
1854,2016-02-26,4087,2713,525,0,4.4950,3.3470,5.5897,False,False,...,False,False,False,False,False,False,False,False,False,1.272119
1855,2016-02-27,5329,3459,600,0,4.4928,3.3441,5.5873,False,False,...,False,False,False,False,False,False,False,False,False,1.221716


Treinando pro Store_Total

In [109]:
y_treino_store = df_resultado_treino['store_total']
X_treino_store= df_resultado_treino.drop(columns= ['store_total', 'FOODS',  'HOBBIES' , 'date' ])


In [110]:
df_resultado_val = pd.merge(df_validacao_interno,df_covariaveis, on= 'date', how = 'inner' )
y_val_store = df_resultado_val['store_total']
X_val_store= df_resultado_val.drop(columns= ['store_total', 'FOODS',  'HOBBIES' , 'date' ])



Modelagem

In [111]:
alphas = [0.01, 0.1 , 1, 10 ]
#  usando LASsp
erros = {alpha: 0 for alpha in alphas}
for alpha in alphas:
    clf =  Lasso(alpha = alpha )
    clf.fit(X_treino_store, y_treino_store)
    y_pred = clf.predict(X_val_store)
    erro_here = mean_absolute_error(y_pred, y_val_store)
    erros[alpha] = erro_here



/home/jaimewillian/Documentos/FGV/6P/Séries Temporais/Series-Temporais-G5/.env/lib/python3.14/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.335620e+08, tolerance: 1.815e+05
  model = cd_fast.enet_coordinate_descent(


In [112]:
melhor_alpha =min(erros, key= lambda x : erros[x])
melhor_alpha

10

Tendo o hiperparametro, vamos a treinamento de verdade

In [ ]:
# continua isolando as datas até o final de fevereiro



clf_definitivo =  Lasso(alpha = melhor_alpha )

# As estatisticas devem ser tiradas do peíodo completo do treino 
mask_treino_total = df_covariaveis['date'] <= '2016-03-27'

# Calcula a média e desvio APENAS no treino
media_foods = df_covariaveis.loc[mask_treino_total, 'preco_FOODS'].mean()
desvio_foods = df_covariaveis.loc[mask_treino_total, 'preco_FOODS'].std()

# Aplica a padronização na coluna inteira (treino e validação)
df_covariaveis['preco_FOODS_padronizada'] = (df_covariaveis['preco_FOODS'] - media_foods) / desvio_foods


df_resultado_treino_total = pd.merge(df_treino,df_covariaveis, on= 'date', how = 'inner' )




y_treino_total = df_resultado_treino_total['store_total']
X_treino_total= df_resultado_treino_total.drop(columns= ['store_total', 'FOODS',  'HOBBIES' , 'date' ])
#  Fazendo a prediçao no treino

clf_definitivo.fit(X_treino_total, y_treino_total)


,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",10
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'
Name,Type,Value


In [ ]:
# y_pred_treino = clf_definitivo.predict(X_val_store)

In [116]:
pacote_modelo = {
    'modelo': clf_definitivo,
    'media_foods': media_foods,
    'desvio_foods': desvio_foods
}

In [117]:
modelo_salvo = joblib.dump(pacote_modelo, 'modelo_salvo.pkl')